# 랭체인 도구(tool)로 에이전트 만들기

도구 호출의 흐름
1. `@tool` 데코레이터로 일반 함수를 '도구'로 등록한다.
2. `llm.bind_tools(tools)` 로 모델에 도구를 알려준다.
3. 모델이 "이 질문엔 도구가 필요하다"고 판단하면, 답변 대신 `response.tool_calls` 에 '어떤 도구를 어떤 인자로 부를지'를 담아준다.
4. 우리가 그 도구를 직접 실행해 `ToolMessage` 로 결과를 붙이고, 다시 모델에 넘기면 도구 결과를 반영한 최종 답변을 만들어 준다.

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
import os

load_dotenv()
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",  # 필수!
    api_key=os.getenv('OPENAI_API_KEY'),       # OpenRouter 전용 키
)

# 도구 없이 일반 질문
llm.invoke([HumanMessage("잘 지냈어?")])

## (1) 도구 정의 — @tool 데코레이터

In [ ]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool  # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} '  # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

## (2) 도구를 모델에 바인딩

In [ ]:
# 도구를 tools 리스트에 추가하고, tool_dict 에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

## (3) 사용자 질문 → 모델이 도구 호출을 요청

In [ ]:
# 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# llm_with_tools 를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# 생성된 llm 답변 출력 (content 는 비어 있고 tool_calls 가 채워진다)
print(messages)

## (4) 도구를 직접 실행해 결과(ToolMessage)를 붙이기

In [ ]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]  # tool_dict 를 사용하여 도구 함수를 선택
    print(tool_call["args"])  # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call)  # 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

In [ ]:
# 도구 결과를 바탕으로 최종 답변 생성
llm_with_tools.invoke(messages)

# 파이단틱 이용하기

도구의 '입력 스키마' 를 파이단틱(`BaseModel`)으로 명시적으로 정의할 수도 있다.

In [ ]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [ ]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown()  # to_markdown() 은 tabulate 패키지가 필요
    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [ ]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

In [ ]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

In [ ]:
llm_with_tools.invoke(messages)

# 스트림 방식으로 출력하기

일반 답변은 조각 단위로 흘러온다.

In [ ]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.")]):
    print(c.content, end="|")

## 스트림 + 도구 호출 — 조각난 tool_call 청크를 하나로 합치기

스트림으로 받을 때 도구 호출 정보(`tool_call`)도 여러 청크로 쪼개져 온다. `AIMessageChunk` 는 `+=` 로 누적해 하나로 합칠 수 있다.

In [ ]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.stream(messages)

# 파편화된 tool_call 청크를 하나로 합치기
is_first = True
for chunk in response:
    print("chunk type: ", type(chunk))

    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk

    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)

In [ ]:
gathered

In [ ]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]  # tool_dict 를 사용하여 도구 이름으로 도구 함수를 선택
    print(tool_call["args"])  # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call)  # 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

In [ ]:
# 도구 결과를 바탕으로 최종 답변을 스트림으로 출력
for c in llm_with_tools.stream(messages):
    print(c.content, end="|")